# Run the backend on Colab GPU

Runs this repo's FastAPI backend (`backend/`) on a Colab GPU runtime and joins it to a
**Tailscale** network, so your **local** frontend (`npm run dev` on your own machine,
unchanged) can call it directly instead of a slow local CPU backend. Unlike an HTTP tunnel
(ngrok/Cloudflare), Tailscale is a direct private network link with no reverse-proxy request
timeout, so it doesn't matter how long a shelf-image upload takes to process.

**Before running:**
1. `Runtime > Change runtime type > GPU`, then `Runtime > Restart session` if you changed it.
2. One-time only: install Tailscale on **this local machine too** (the one running `npm run
   dev`) from https://tailscale.com/download and log in — both it and the Colab VM need to be
   on the same (free) tailnet for this to work.
3. One-time only: upload `backend/models/best_augmented.pt` and everything in `backend/data/`
   (`class_keywords.json`, `embeddings.json`, ...) from your machine into a Google Drive folder
   at `MyDrive/lipton-sku-classifier-assets/models/` and `MyDrive/lipton-sku-classifier-assets/data/`
   respectively. These files are gitignored (see `backend/.gitignore`) so they don't come from
   `git clone` below — Drive is just the hand-off point between your machine and the Colab VM.
4. One-time only: accept the license for the gated `facebook/dinov3-vitb16-pretrain-lvd1689m`
   model on Hugging Face and grab an access token — see the "Hugging Face login" cell below for
   the exact steps.
5. One-time only: generate a Tailscale auth key — see the "Tailscale auth" cell below for the
   exact steps.
6. Run every cell top to bottom. The last cell prints the URL to paste into your local
   `frontend/.env` as `VITE_API_BASE`.

The backend keeps running as a background process as long as this notebook's runtime stays
alive — closing the tab eventually disconnects the runtime (Colab free tier), which kills it,
so re-run the notebook to get a new session.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU, restart, and re-run."
    )

In [ ]:
REPO_URL = "https://github.com/ansshahzadd/lipton-sku-classifier.git"
REPO_DIR = "/content/lipton-sku-classifier"
BACKEND_DIR = f"{REPO_DIR}/backend"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

In [ ]:
# Install everything from requirements.txt (this pulls in the CPU paddlepaddle build), then
# try to swap in the matching GPU build so OCR can run on the GPU too.
#
# PaddlePaddle 3.x unified the package name -- there is no separate "paddlepaddle-gpu"
# distribution any more, even though Baidu's package index still uses that as a URL folder
# name; every wheel actually served under .../cuXXX/paddlepaddle-gpu/ is named
# "paddlepaddle-3.x...", not "paddlepaddle_gpu-...".
#
# That folder is also NOT a real PEP 503 index (a directory-per-package structure), just a
# plain file listing -- so `-i/--index-url` (which makes pip auto-append the requested
# package's own name as a subpath, e.g. request "paddlepaddle" -> fetch ".../paddlepaddle/",
# which is empty) can never reach the real wheels, regardless of what package name you ask
# for. `-f/--find-links` fixes this: it scans exactly the page you give it for matching
# wheel links instead of doing that path-joining. Verified with `pip download` against the
# real cu128 listing before wiring this in.
#
# --force-reinstall is needed because requirements.txt already installed a CPU "paddlepaddle"
# moments ago, and an unpinned `pip install paddlepaddle` would otherwise treat that as already
# satisfying the requirement and do nothing; --no-deps avoids pip trying to also resolve
# paddlepaddle's other dependencies against this paddle-only page (they're already installed
# from the earlier PyPI-based requirements.txt install).
#
# We derive the wheel tag directly from Colab's reported CUDA version (e.g. "12.8" -> "cu128")
# rather than a fixed lookup table -- verified the index publishes real wheels for cu118
# through cu129, so this covers whatever CUDA version Colab hands us. If the derived tag
# doesn't exist there, we keep the CPU paddle and just run OCR on CPU (YOLO + DINOv3 still get
# the full GPU speedup either way).
%pip install -q -r {BACKEND_DIR}/requirements.txt

import subprocess
import torch

cuda_version = torch.version.cuda  # e.g. "12.8"
cuda_tag = f"cu{cuda_version.replace('.', '')}" if cuda_version else None
print("torch reports CUDA", cuda_version, "-> paddle wheel tag:", cuda_tag)

paddle_gpu_ok = False
if cuda_tag:
    result = subprocess.run(
        ["pip", "install", "-q", "--force-reinstall", "--no-deps", "paddlepaddle",
         "-f", f"https://www.paddlepaddle.org.cn/packages/stable/{cuda_tag}/paddlepaddle-gpu/"],
        capture_output=True, text=True,
    )
    paddle_gpu_ok = result.returncode == 0
    if not paddle_gpu_ok:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])

print("paddlepaddle-gpu installed:", paddle_gpu_ok, "-> OCR will run on", "GPU" if paddle_gpu_ok else "CPU")

In [ ]:
# DINOv3 (facebook/dinov3-vitb16-pretrain-lvd1689m) is a GATED model on Hugging Face -- pipeline.py
# downloads it on first run, and that download 401s until you've (1) accepted its license and
# (2) authenticated here with a token. One-time setup, from your own machine/browser:
#   1. Log into https://huggingface.co (free account is fine).
#   2. Open https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m and click
#      "Agree and access repository" (Meta's DINOv3 grants are usually instant).
#   3. Create a read-scope token at https://huggingface.co/settings/tokens.
# Then either: add it as a Colab secret named HF_TOKEN (key icon in the left sidebar, toggle
# "Notebook access" on) so this cell picks it up with no prompt, or just paste it when asked
# below (input is hidden either way, nothing gets printed or saved into this notebook).
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    import getpass
    hf_token = getpass.getpass("Hugging Face token (read scope): ")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Logged in to Hugging Face.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

ASSETS_DIR = "/content/drive/MyDrive/lipton-sku-classifier-assets"

for sub in ("models", "data"):
    src = os.path.join(ASSETS_DIR, sub)
    dst = os.path.join(BACKEND_DIR, sub)
    if not os.path.isdir(src):
        raise FileNotFoundError(
            f"Expected {src} on your Drive. Upload your {sub}/ files there first (see the "
            "instructions in the first cell), then re-run this cell."
        )
    os.makedirs(dst, exist_ok=True)
    for fname in os.listdir(src):
        shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
    print(f"{sub}/ ->", os.listdir(dst))

In [ ]:
!curl -fsSL https://tailscale.com/install.sh | sh

In [ ]:
# There's no systemd in Colab, so tailscaled (the background daemon `tailscale` talks to) has
# to be started and supervised manually, same as uvicorn later in this notebook.
#
# Colab's runtime is a sandboxed container with no /dev/net/tun and no CAP_NET_ADMIN (even as
# root, iptables/TUN-device creation are blocked) -- so tailscaled can't set up a normal kernel
# network interface. --tun=userspace-networking makes it use its own userspace TCP/IP stack
# instead, which needs neither; inbound connections from your tailnet to services on this VM
# (like uvicorn on :8000 later) still get forwarded through it correctly.
import os, subprocess, threading, time

if "tailscaled_proc" in globals():
    try:
        tailscaled_proc.terminate()
        tailscaled_proc.wait(timeout=5)
    except Exception:
        pass
!pkill -f "tailscaled" 2>/dev/null || true
time.sleep(1)

os.makedirs("/var/lib/tailscale", exist_ok=True)
os.makedirs("/run/tailscale", exist_ok=True)
TS_SOCKET = "/run/tailscale/tailscaled.sock"

tailscaled_proc = subprocess.Popen(
    ["tailscaled", "--tun=userspace-networking",
     "--state=/var/lib/tailscale/tailscaled.state", f"--socket={TS_SOCKET}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def _watch_tailscaled_output():
    for line in tailscaled_proc.stdout:
        print(line, end="")

threading.Thread(target=_watch_tailscaled_output, daemon=True).start()
time.sleep(2)
print("tailscaled starting (pid", tailscaled_proc.pid, ")")

In [ ]:
# Join your tailnet -- one-time setup, from your own browser:
#   1. Sign up free at https://login.tailscale.com/start (if you haven't already for the local
#      install mentioned in the first cell).
#   2. Generate an auth key at https://login.tailscale.com/admin/settings/keys -> "Generate auth
#      key..." -> enable both "Reusable" and "Ephemeral" (ephemeral means this Colab VM's node
#      disappears from your tailnet automatically when the session ends, instead of piling up
#      dead devices every time you re-run this notebook).
# Then either: add it as a Colab secret named TAILSCALE_AUTHKEY, or paste it when asked below
# (hidden input, nothing gets printed or saved into this notebook).
ts_key = None
try:
    from google.colab import userdata
    ts_key = userdata.get("TAILSCALE_AUTHKEY")
except Exception:
    ts_key = None

if not ts_key:
    import getpass
    ts_key = getpass.getpass("Tailscale auth key: ")

!tailscale --socket=/run/tailscale/tailscaled.sock up --authkey={ts_key} --hostname=colab-backend --accept-routes

In [ ]:
import json, subprocess

status = json.loads(subprocess.run(
    ["tailscale", "--socket=/run/tailscale/tailscaled.sock", "status", "--json"],
    capture_output=True, text=True, check=True,
).stdout)

dns_name = status["Self"]["DNSName"].rstrip(".")
public_url = f"http://{dns_name}:8000"
print("Backend address (reachable only from devices on your tailnet):", public_url)

In [ ]:
import os, subprocess, threading, time

# Kill any backend left over from a previous run of this cell -- otherwise the new uvicorn
# fails to bind port 8000 ("address already in use") and silently exits, while the old,
# possibly-still-busy process keeps eating every request.
if "backend_proc" in globals():
    try:
        backend_proc.terminate()
        backend_proc.wait(timeout=5)
    except Exception:
        pass
!pkill -f "uvicorn main:app" 2>/dev/null || true
time.sleep(1)

env = os.environ.copy()
env["PUBLIC_BASE_URL"] = public_url
env["FRONTEND_ORIGINS"] = "http://localhost:5173,http://127.0.0.1:5173"
env["OCR_DEVICE"] = "gpu" if paddle_gpu_ok else "cpu"

backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=BACKEND_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def _watch_backend_output():
    for line in backend_proc.stdout:
        print(line, end="")

threading.Thread(target=_watch_backend_output, daemon=True).start()
print("uvicorn starting (pid", backend_proc.pid, ") — first request will be slow while models load.")

In [ ]:
import time, urllib.request

up = False
for _ in range(90):
    try:
        with urllib.request.urlopen(public_url + "/api/dashboard", timeout=3) as resp:
            if resp.status == 200:
                up = True
                break
    except Exception:
        pass
    time.sleep(2)

if not up:
    print("Backend didn't respond in time — check the uvicorn logs in the cell above for errors.")
else:
    print("Backend is up and reachable through the tunnel.\n")

print("=" * 60)
print("On your LOCAL machine (with Tailscale connected), set this in frontend/.env, then")
print("`npm run dev`:")
print(f"VITE_API_BASE={public_url}")
print("=" * 60)

## Notes

- OCR device is picked automatically: the install cell tries to match Colab's CUDA version to a
  `paddlepaddle-gpu` wheel from Baidu's package index and sets `paddle_gpu_ok` accordingly; the
  uvicorn cell reads that to set `OCR_DEVICE`. If it falls back to CPU, YOLO + DINOv3 still get
  the full GPU speedup — only OCR runs on CPU.
- We use Tailscale instead of an HTTP tunnel (ngrok/Cloudflare) because both of those force-
  close a request after a fixed time (~100s for Cloudflare's free quick tunnel, ~300s for
  ngrok's free tier) — a slow multi-crop shelf-image upload can exceed either one. Tailscale is
  a direct private network link (WireGuard-based), not an HTTP reverse proxy, so it has no such
  cap. The tradeoff: your local machine must also be on the same tailnet and connected whenever
  you want to reach the backend (it's a private address, not a public URL).
- The backend's address is a stable MagicDNS name (`http://colab-backend.<your-tailnet>.ts.net:8000`)
  as long as you keep `--hostname=colab-backend` in the auth cell, so unlike ngrok/Cloudflare
  URLs it doesn't change between sessions — you should only need to set `VITE_API_BASE` once.
- Re-running the daemon or uvicorn cell kills whatever it previously started first, so it's
  safe to re-run either one on its own without restarting the whole runtime.
- A single `/api/images` upload can legitimately take a while (many crops per shelf photo, each
  doing an embedding + up to 2 OCR passes) — `main.py` runs it in a background thread
  (`asyncio.to_thread`) so it no longer freezes the whole server for other requests while it's
  in flight.
- To stop everything: `backend_proc.terminate(); tailscaled_proc.terminate()`, or just
  stop/disconnect the Colab runtime.